# The Vibe Tax — LCB extractor (standalone)

A self-contained notebook to read the code-extraction logic line by line and step
through LiveCodeBench problems one at a time. **No git clone, no repo** — you upload
the data files and everything else is defined inline.

**Files to upload (Step 1):**
| file | required? | gives |
|------|-----------|-------|
| `lcb_v3_responses.json` | **yes** | the model replies to extract from (also carries `entry_point` + the prompt) |
| `lcb_problems.jsonl` | optional | clean problem statement + difficulty (else the wrapped prompt is shown) |
| `lcb_tests.jsonl` | optional (**large ~400MB**) | real graded tests → per-test PASS/FAIL. Skip for a code walkthrough. |

Detection is by **content**, so filenames don't have to match exactly.

## Step 1 · Upload your JSON files
Run this, pick `lcb_v3_responses.json` (required) and optionally the others. It figures
out which is which and loads them into `RESP`, `PROBS`, `TESTS`.

In [ ]:
import json
try:
    from google.colab import files
    up = files.upload()                      # pick 1-3 files
except Exception:
    up = {}                                  # (running outside Colab) place files next to the notebook
    for n in ("lcb_v3_responses.json","lcb_problems.jsonl","lcb_tests.jsonl"):
        if __import__("os").path.exists(n):
            up[n] = open(n,"rb").read()

PROBS, RESP, TESTS = {}, {}, {}
for name, data in up.items():
    text = data.decode("utf-8", errors="replace")
    s = text.lstrip()
    if s.startswith("["):                                   # JSON array -> responses
        recs = json.loads(text)
        RESP = {(r["task_id"], r["level"], r["model"]): r for r in recs}
        print(f"[responses] {name}: {len(recs)} completions")
    else:                                                   # JSONL -> problems or tests
        rows = [json.loads(l) for l in text.splitlines() if l.strip()]
        first = rows[0] if rows else {}
        if "public_tests" in first or "private_tests" in first:
            TESTS = {d["task_id"]: d for d in rows}
            print(f"[tests] {name}: {len(TESTS)} problems")
        else:
            PROBS = {d["task_id"]: d for d in rows}
            print(f"[problems] {name}: {len(PROBS)} problems")

assert RESP, "lcb_v3_responses.json is required — re-run and upload it."
print(f"\nready: RESP={len(RESP)}  PROBS={len(PROBS)}  TESTS={len(TESTS)}"
      + ("   (no tests -> graded PASS/FAIL disabled, extraction still works)" if not TESTS else ""))

## Step 2 · The extractor + scorer (defined inline)
This is the exact code from `score_lcb.py`, pasted here so the notebook is standalone.
Run it once; the walkthrough below explains each function, and Step 3 uses them.

In [ ]:
# --- extractor + scorer, copied verbatim from score_lcb.py (standalone) ---
import ast, json, re, time, multiprocessing, inspect

_IMPORTS = 'from typing import *\nimport collections, math, heapq, bisect, itertools, functools, re\nfrom collections import *\nfrom math import *\nfrom functools import *\n'
PER_TEST_TIMEOUT = 6   # seconds allowed per test
PRIVATE_ONLY = False   # sample_tests: include public tests too

def parse_lit(s):
    if not isinstance(s, str):
        return s
    s = s.strip()
    for fn in (json.loads, ast.literal_eval):
        try:
            return fn(s)
        except Exception:
            continue
    return s  # leave as raw string


def parse_input(inp):
    """LCB functional input = one literal per line -> list of args."""
    if isinstance(inp, list):
        return [parse_lit(x) for x in inp]
    return [parse_lit(line) for line in str(inp).split("\n") if line.strip() != ""]


def _trim_to_compilable(code):
    """Drop trailing lines until the source parses. Conversational replies put a
    prose paragraph AFTER the code ('This checks every adjacent pair...'); with no
    fences to delimit it, that prose used to be exec'd and threw SyntaxError,
    failing correct code. The polite/detailed framing elicits more explain-after-
    code, so this bug penalized it hardest — a scoring artifact, not correctness.
    We keep the largest leading prefix that is valid Python and still has the
    class/method."""
    lines = code.split("\n")
    while lines:
        src = "\n".join(lines)
        try:
            ast.parse(src)
            return src
        except SyntaxError:
            lines.pop()
    return None


def extract_solution(completion, entry):
    """Pull runnable code that defines `class Solution` (or the method), robust to
    trailing prose, leading chatter, AND helper definitions (Fenwick/DSU/Segments…)
    the model defines BEFORE `class Solution`. Returns the first candidate that
    compiles and still defines the target. Candidates that KEEP the helpers are
    tried first, so we don't hand exec() a Solution that references an undefined
    helper we chopped off."""
    if not completion:
        return None
    text = "\n".join(l for l in completion.split("\n") if not l.strip().startswith("```"))
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", completion, re.DOTALL)
    code_blocks = [b for b in blocks if re.search(r"(?m)^\s*(?:class|def|import|from|@)\s", b)]

    cands = []
    # (B) from the FIRST top-level code construct to the end — keeps helper
    #     classes/functions defined above `class Solution` (fixes the drop bug).
    m = re.search(r"(?m)^(?:class|def|import|from|@)\s", text)
    if m:
        cands.append(text[m.start():])
    # (A) all fenced code blocks concatenated — helper and Solution in separate blocks.
    if code_blocks:
        cands.append("\n\n".join(code_blocks))
    # (C) a single block that names the target (old high-confidence path).
    for b in blocks:
        if "class Solution" in b or f"def {entry}" in b:
            cands.append(b)
    # (D) old anchor slice, last resort.
    for anchor in ("class Solution", f"def {entry}"):
        i = text.find(anchor)
        if i != -1:
            cands.append(text[i:])

    for c in cands:
        t = _trim_to_compilable(c)
        if t and ("class Solution" in t or f"def {entry}" in t):
            return t
    return cands[0] if cands else None


def resolve_callable(ns, entry):
    """Find the `entry` implementation regardless of how the model SHAPED its code.

    LCB expects `Solution().<entry>(*args)`. But a correct solution is sometimes
    delivered in a different container — a bare module-level function, a method on a
    RENAMED top-level class, or a method on a class nested one level inside another.
    Those are code *smells*, not wrong answers, yet the old resolver (hardcoded
    `ns['Solution']`) scored them FAIL. We look past the shape and bind to the actual
    implementation. Returns a callable (bound, if a method) or None.

    Order: (1) a bare module-level function named `entry`; else (2) a class exposing
    a callable `entry`, preferring one literally named `Solution`, searching both
    top-level classes and classes nested one level inside another class."""
    import inspect
    f = ns.get(entry)
    if callable(f) and not inspect.isclass(f):
        return f                                   # bare function shape
    classes = [v for v in ns.values() if inspect.isclass(v)]
    nested = [v for c in classes for v in vars(c).values() if inspect.isclass(v)]
    cands = [c for c in (classes + nested) if callable(getattr(c, entry, None))]
    if not cands:
        return None
    cands.sort(key=lambda c: c.__name__ != "Solution")   # prefer the expected name
    try:
        inst = cands[0]()
    except Exception:
        return None
    return getattr(inst, entry, None)


def eq(a, b):
    if isinstance(a, float) or isinstance(b, float):
        try:
            return abs(float(a) - float(b)) < 1e-6
        except Exception:
            return False
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(eq(x, y) for x, y in zip(a, b))
    return a == b


def _worker(code, entry, tests, q):
    """Run tests in order, STREAMING one bool per test into q, and stop at the
    first failure (early exit). Streaming lets passes() enforce a PER-TEST timeout
    instead of one budget for the whole suite — a correct-but-slow solution on a
    many-test problem must not be failed just because the SUM of test times exceeds
    a single cap (LCB, like any judge, limits each test, not the total)."""
    ns = {}
    try:
        exec(_IMPORTS + code, ns)
    except Exception:
        q.append(False); return
    fn = resolve_callable(ns, entry)               # shape-agnostic (class/bare/renamed/nested)
    for t in tests:
        try:
            args = parse_input(t["input"])
            expected = parse_lit(t["output"])
            ok = (fn is not None) and eq(fn(*args), expected)
        except Exception:
            ok = False
        q.append(ok)
        if not ok:
            return                      # early exit: one failure fails the problem


def passes(code, entry, tests):
    """True iff EVERY test passes, each within PER_TEST_TIMEOUT seconds. A wrong
    answer fails fast; a test that hangs (infinite loop / TLE) fails on its own
    timeout without killing the tests that already passed."""
    if not code or not tests:
        return False
    mgr = multiprocessing.Manager(); q = mgr.list()
    p = multiprocessing.Process(target=_worker, args=(code, entry, tests, q))
    p.start()
    seen = 0; last_progress = time.time()
    while p.is_alive():
        n = len(q)
        if n > seen:
            if q[n - 1] is False:                 # newest test failed -> stop
                p.kill(); p.join(2); return False
            seen = n; last_progress = time.time()
        elif time.time() - last_progress > PER_TEST_TIMEOUT:   # hung on current test
            p.kill(); p.join(2); return False
        time.sleep(0.02)
    res = list(q)
    return len(res) == len(tests) and all(res)    # all tests present and passed


def sample_tests(rec, k):
    pub = [] if PRIVATE_ONLY else rec.get("public_tests", [])
    priv = rec.get("private_tests", [])
    if len(pub) + len(priv) <= k:
        return pub + priv
    need = max(0, k - len(pub))
    stride = max(1, len(priv) // need) if need else 1
    return pub + (priv[::stride][:need] if need else [])


print("extractor + scorer defined:",
      [f for f in ("extract_solution","_trim_to_compilable","resolve_callable","passes")])

## Walkthrough — line by line
Chain per completion: `extract_solution` (get code) → exec + `resolve_callable`
(find the function) → per-test compare → `passes` (per-test timeout + all-must-pass).

### (1) `_trim_to_compilable` — strip trailing prose so correct code parses
```python
def _trim_to_compilable(code):
    lines = code.split("\n")     # split the reply text into individual lines
    while lines:                  # loop while any lines remain
        src = "\n".join(lines)   # rejoin the current lines into one string
        try:
            ast.parse(src)        # can Python PARSE this? (syntax check, does NOT run it)
            return src            # yes -> largest valid-Python prefix; return it
        except SyntaxError:
            lines.pop()           # no -> drop the LAST line (trailing prose) and retry
    return None                   # nothing ever parsed -> no usable code
```
Replies are often `...correct code... + an English paragraph`; the paragraph makes
`ast.parse` throw, so we peel lines off the bottom until the rest parses.

### (2) `extract_solution` — pull runnable code out of a messy reply
```python
def extract_solution(completion, entry):
    if not completion:                       # empty/None reply
        return None
    text = "\n".join(l for l in completion.split("\n")
                      if not l.strip().startswith("```"))   # copy with fence MARKERS removed
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", completion, re.DOTALL)  # fenced blocks
    code_blocks = [b for b in blocks
                   if re.search(r"(?m)^\s*(?:class|def|import|from|@)\s", b)]  # keep code-like blocks

    cands = []                               # ordered candidate programs to try
    m = re.search(r"(?m)^(?:class|def|import|from|@)\s", text)   # first top-level construct
    if m:
        cands.append(text[m.start():])       # (B) slice from there -> KEEPS helpers above Solution
    if code_blocks:
        cands.append("\n\n".join(code_blocks))  # (A) all code blocks joined
    for b in blocks:                         # (C) any single block naming the target
        if "class Solution" in b or f"def {entry}" in b:
            cands.append(b)
    for anchor in ("class Solution", f"def {entry}"):   # (D) last resort: slice from anchor
        i = text.find(anchor)
        if i != -1:
            cands.append(text[i:])

    for c in cands:                          # try candidates IN ORDER
        t = _trim_to_compilable(c)           #   trim each to its compilable prefix
        if t and ("class Solution" in t or f"def {entry}" in t):  # compiles AND still has target
            return t                         #   accept the first that qualifies
    return cands[0] if cands else None       # graceful fallback (never a spurious None)
```
Never invents correctness — the returned code still has to pass the real tests.

### (3) `resolve_callable` — find the function whatever container it's in
```python
def resolve_callable(ns, entry):             # ns = namespace after exec'ing the code
    import inspect
    f = ns.get(entry)                         # (1) a top-level function literally named entry?
    if callable(f) and not inspect.isclass(f):
        return f                              #     yes -> bare-function shape
    classes = [v for v in ns.values() if inspect.isclass(v)]        # top-level classes
    nested  = [v for c in classes for v in vars(c).values()
               if inspect.isclass(v)]                                # classes nested 1 level in
    cands = [c for c in (classes + nested)
             if callable(getattr(c, entry, None))]                   # classes exposing entry()
    if not cands:
        return None
    cands.sort(key=lambda c: c.__name__ != "Solution")  # prefer a class named 'Solution'
    try:
        inst = cands[0]()                     # instantiate (LCB Solution takes no args)
    except Exception:
        return None
    return getattr(inst, entry, None)         # return the BOUND method
```
Handles a correct answer delivered as a bare function, a renamed class, or a nested class.

### (4) `_worker` + `passes` — run the tests safely, per-test timeout
```python
def _worker(code, entry, tests, q):          # runs in a SEPARATE process
    ns = {}
    try:
        exec(_IMPORTS + code, ns)             # run the code (+common imports) into ns
    except Exception:
        q.append(False); return               # won't import -> one failure, stop
    fn = resolve_callable(ns, entry)          # get the entry function, any shape
    for t in tests:
        try:
            args = parse_input(t["input"])    # "1\n[2,3]" -> [1, [2,3]] (one literal per line)
            expected = parse_lit(t["output"]) # parse expected return as a Python literal
            ok = (fn is not None) and eq(fn(*args), expected)  # call + compare (float tolerant)
        except Exception:
            ok = False
        q.append(ok)                          # STREAM this test's result out immediately
        if not ok:
            return                            # early exit: one bad test fails the problem

def passes(code, entry, tests):
    if not code or not tests:
        return False
    mgr = multiprocessing.Manager(); q = mgr.list()   # list shared between processes
    p = multiprocessing.Process(target=_worker, args=(code, entry, tests, q))
    p.start()                                 # launch worker (a crash/loop can't kill us)
    seen = 0; last_progress = time.time()
    while p.is_alive():
        n = len(q)
        if n > seen:                          # worker reported new result(s)
            if q[n - 1] is False:             #   newest is a failure
                p.kill(); p.join(2); return False
            seen = n; last_progress = time.time()   # progress -> reset per-test clock
        elif time.time() - last_progress > PER_TEST_TIMEOUT:  # no result for too long
            p.kill(); p.join(2); return False #   current test hung (TLE) -> fail
        time.sleep(0.02)
    res = list(q)
    return len(res) == len(tests) and all(res)  # every test present AND all passed
```
Each test independently gets `PER_TEST_TIMEOUT`s, so correct-but-slow isn't punished by
one global budget, and a hang is killed without freezing the notebook.

## Step 3 · Inspect ONE problem
Run this to define `inspect_problem(...)`, then use the **call cell** — change `task_id`
(and optionally `condition`/`model`/`custom_input`) and re-run to step through problems.

In [ ]:
import multiprocessing

def _run_one_test(code, entry, case, timeout=6):
    """Run ONE test in a subprocess; return (status, got_repr). Safe against hangs."""
    def worker(q):
        ns = {}
        try:
            exec(_IMPORTS + code, ns)
        except Exception as e:
            q.append(("error", f"exec failed: {type(e).__name__}: {e}")); return
        fn = resolve_callable(ns, entry)
        if fn is None:
            q.append(("error", "entry not found")); return
        try:
            got = fn(*parse_input(case["input"]))
            q.append(("pass" if eq(got, parse_lit(case["output"])) else "wrong", repr(got)))
        except Exception as e:
            q.append(("error", f"{type(e).__name__}: {e}"))
    mgr = multiprocessing.Manager(); q = mgr.list()
    p = multiprocessing.Process(target=worker, args=(q,)); p.start(); p.join(timeout)
    if p.is_alive():
        p.kill(); p.join(1); return ("timeout", f">{timeout}s (loop/TLE)")
    return q[0] if q else ("error", "worker crashed")

def inspect_problem(task_id, condition="agentic_terse", model="chatgpt",
                    n_show=None, only_fails=False, width=40, max_tests=60,
                    custom_input=None, show_raw=False):
    r = RESP.get((task_id, condition, model))
    if not r:
        if not any(k[0] == task_id for k in RESP):
            print("unknown task_id:", task_id, "| e.g.", sorted({k[0] for k in RESP})[:5]); return
        print("no completion for that (task, condition, model).")
        print("  conditions here:", sorted({k[1] for k in RESP if k[0]==task_id}))
        print("  models here    :", sorted({k[2] for k in RESP if k[0]==task_id}))
        return
    entry = r["entry_point"]
    pinfo = PROBS.get(task_id, {})
    statement = pinfo.get("question_content") or r.get("prompt_text", "(upload lcb_problems.jsonl for the clean statement)")
    print(f"TASK {task_id} | difficulty={pinfo.get('difficulty','?')} | entry={entry}")
    print(f"condition={condition} | model={model}")
    print("-"*72)
    print("PROBLEM (first 500 chars):\n", statement[:500], "...\n")
    if show_raw:
        print("RAW MODEL REPLY:\n", r["completion"], "\n" + "-"*72)
    code = extract_solution(r["completion"], entry)
    print("EXTRACTED CODE:\n", code, "\n" + "-"*72)
    ns = {}
    try:
        exec(_IMPORTS + code, ns)
        fn = resolve_callable(ns, entry)
        print("RESOLVED CALLABLE:", fn)
    except Exception as e:
        print("EXTRACTED CODE FAILS TO EXEC:", type(e).__name__, e); return
    if custom_input is not None:
        try:
            print(f"\nCUSTOM INPUT {custom_input!r}  ->  {fn(*parse_input(custom_input))!r}")
        except Exception as e:
            print("custom call error:", type(e).__name__, e)
    rec = TESTS.get(task_id)
    if not rec:
        print("\n(no lcb_tests.jsonl -> graded tests skipped; upload it to enable)")
        return
    from collections import Counter
    cases = sample_tests(rec, max_tests)
    tally = Counter()
    print(f"\nGRADED TESTS ({len(cases)} run"
          + (", showing only failures" if only_fails else "") + "):")
    for i, c in enumerate(cases):
        status, got = _run_one_test(code, entry, c)
        tally[status] += 1
        if (n_show is None or i < n_show) and (not only_fails or status != "pass"):
            inp = c["input"] if isinstance(c["input"], str) else str(c["input"])
            inp = inp.replace("\n", " | ")                     # flatten multi-line input to one row
            print(f"  #{i:3d} {status:7s} in={inp[:width]:<{width}} "
                  f"exp={str(c['output'])[:width]:<{width}} got={got[:width]}")
    npass = tally.get("pass", 0)
    print("\n  tally: " + "  ".join(f"{k}={v}" for k, v in sorted(tally.items())))
    print(f"  RESULT: {npass}/{len(cases)} passed  ->  {'PASS' if npass==len(cases) else 'FAIL'}")

print("inspect_problem() ready — edit the call cell below and re-run.")

### ▶ Call cell — edit and re-run per problem

In [ ]:
inspect_problem(
    task_id   = "lcb/3517",          # <-- change to any id (see the browse cell below)
    condition = "agentic_terse",     # agentic_terse | agentic_casual | webchat_detailed | webchat_multilingual
    model     = "chatgpt",           # chatgpt | claude | codestral
    n_show    = None,                # None = EVERY test row; or an int for just the first N
    only_fails = False,              # True = print only the tests that did NOT pass
    width     = 40,                  # column width for input / expected / got
    max_tests = 60,                  # how many tests to actually run (raise to run more)
    custom_input = None,             # e.g. "5\n[1,2,3]" to call the solution with your own args
    show_raw  = False,               # True to also print the raw model reply
)

### Browse available problem ids

In [ ]:
ids = PROBS if PROBS else {k[0]: {} for k in RESP}
print(f"{len(ids)} problems")
print(f"{'task_id':10s} {'difficulty':10s} entry_point")
for tid in sorted(ids):
    d = PROBS.get(tid, {})
    ep = d.get("entry_point") or next((r["entry_point"] for k, r in RESP.items() if k[0]==tid), "")
    print(f"{tid:10s} {str(d.get('difficulty','?')):10s} {ep}")